In [ ]:
""""
Tasks:
• Use OpenCV Haar cascades for face detection
• Use dlib or MediaPipe for 68-point landmarks
• Crop and align faces to a standard size
• Visualise landmarks on 10 real images
• Save aligned face crops as a dataset
Tools: OpenCV, dlib or MediaPipe, numpy, matplotlib
Deliverable: Script that takes any image and outputs aligned face crop with landmarks drawn
 Tip: Face alignment (rotating so eyes are horizontal) significantly improves deepfake detection accuracy.
 """

In [2]:
import cv2

# Accessing Webcam

In [ ]:
# video_capture=cv2.VideoCapture(0)
# while True:
#     ret,video_data=video_capture.read()
#     cv2.imshow("video_live",video_data)
#     if (cv2.waitKey(10)==ord('a')):
#         break
# video_capture.release()    
# cv2.destroyAllWindows()

In [ ]:
video_cap=cv2.VideoCapture(0)
# 0 -> opens the default cam
while True:
    ret, video_data=video_cap.read()
    cv2.imshow("Umyma's Video",video_data)
    # waitkey pauses the program briefly anc captures input
    if(cv2.waitKey(20)==ord("c")):
        break
video_cap.release() 
# kernel crashes if the window is not destroyed
cv2.destroyAllWindows()

# Face Detection

In [ ]:
face_cap=cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
video_capture=cv2.VideoCapture(0)
while True:
    ret, vid_data=video_capture.read()
    col=cv2.cvtColor(vid_data,cv2.COLOR_BGR2GRAY)
    face=face_cap.detectMultiScale(
        col,
        scaleFactor=1.3,
        minNeighbors=5,
        minSize=(30,30),
        flags=cv2.CASCADE_SCALE_IMAGE)
    # print("Faces detected:", len(face))
    for (x,y,w,h) in face:
        cv2.rectangle(vid_data,(x,y),(x+w,y+h),(0,255,0),2)
    cv2.imshow("Video",vid_data)
    if (cv2.waitKey(10)==ord("a")):
        # break if "a" is pressed for 10 mil secs
        break
video_capture.release()
cv2.destroyAllWindows()   

In [1]:
import cv2
import mediapipe as mp

In [3]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe import Image, ImageFormat

In [5]:
import os
import urllib

Landmark Extraction

In [6]:
model_path = "face_landmarker.task"
if not os.path.exists(model_path):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
        model_path
    )

In [15]:
#  creating a facelandmarker object

base_options=python.BaseOptions(model_asset_path=model_path)
options=vision.FaceLandmarkerOptions(base_options=base_options,
                                    output_face_blendshapes=True, #detects facial expressions
                                    output_facial_transformation_matrixes=True, #calculates 3d position of the face in space
                                    num_faces=1) #looks for only one face at a time
detector=vision.FaceLandmarker.create_from_options(options=options)
# detector object refers to the pretrained model

In [16]:
# getting frames from webcam
cap=cv2.VideoCapture(0) #captures 30 frames/sec
while True:
    ret,frame=cap.read() #frame contains the caotured img as an np array
    frame=cv2.flip(frame,1)
    # flipping the frame horizontally (mirrored view)
    rgb_frame=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB) #swapping the color channel order
    # cv2 reads images in gbr order whereas mp expects rgb
    mp_image=Image(image_format=ImageFormat.SRGB,data=rgb_frame) 
    # np img array -> mp img obj
    results=detector.detect(mp_image)
    if results.face_landmarks:
        h,w=frame.shape[:2]
        # gets the h and w of the frame
        for face_landmarks in results.face_landmarks:
            # loops over the detected face (1)
            for landmark in face_landmarks:
                # loops over 478 landmark points
                x = int(landmark.x * w)
                y = int(landmark.y * h)
                cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)
    cv2.imshow("Face Mesh",frame)
    if cv2.waitKey(10)==ord('a'):
        break            
cap.release()
detector.close()
cv2.destroyAllWindows()
